# Emotion Detection

Given an image of a facial expression, we want to determine what emotion is being portrayed. This is a multiclass classification problem as there are a total of 7 different emotions that we can classify instances as: angry, disgust, fear, happy, sad, surprised, and neutral. Our goal is to build and compare different Convolutional Neural Network (CNN) models to classify these emotions from inputted images.

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.utils import to_categorical
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, BatchNormalization, MaxPooling2D, Dropout, Flatten, Dense
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import pickle

## Load Preprocessed Data

In [ ]:
data = np.load('../data/processed_data.npz')

X_train = data['X_train']
X_test = data['X_test']
X_val = data['X_val']

X_train_normalized = data['X_train_normalized']
X_test_normalized = data['X_test_normalized']
X_val_normalized = data['X_val_normalized']

y_train = data['y_train']
y_test = data['y_test']
y_val = data['y_val']

y_train_cat = data['y_train_cat']
y_test_cat = data['y_test_cat']
y_val_cat = data['y_val_cat']

## Data Augmentation

There is a significant difference in the number of images across the different emotion classes. This could potentially lead to overfitting as our model will be trained using # layer that randomly applies transformations (flips, rotations, zooms) to each input image during training only
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])more images from certain classes than others. In order to get more data to use to train our model and to fix the imbalance, we will oversample the training data by performing image augmentation. We choose to oversample instead of undersample as undersampling can lead to the loss of important data. We also do not want to duplicate images as this can also lead to overfitting.

Perform data augmentation on all classes (not just minority ones) to create new versions of existing images. This teaches the model to handle small variations in images.

In [ ]:
# layer that randomly applies transformations (flips, rotations, zooms) to each input image during training only
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

## Build Models


Build a Convolutional Neural Network (CNN) that classifies images by 1 of 7 emotion types.

### Model 1 (Baseline)

Start by creating a simple model with only 1 convolutional layer.

In [ ]:
model1 = tf.keras.Sequential([
    # define input shape, 48x48 pixels, 1 channel (grayscale)
    layers.Input(shape=(48, 48, 1)),

    # convolutional layer
    layers.Conv2D(32, (3, 3), activation='relu'),

    # convert 2D feature map into a 1D vector
    layers.Flatten(),

    # fully connected layer that learns combinations of the features detected by the convolutional layer, 64 neurons = 64 combinations/patterns
    layers.Dense(64, activation='relu'),

    # final layer with 7 neurons (one for each emotion)
    layers.Dense(7, activation='softmax')
])

model1.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history1 = model1.fit(
    X_train_normalized, y_train_cat,
    epochs=30,
    batch_size=64,
    validation_data=(X_val_normalized, y_val_cat),
    callbacks=[early_stop]
)

# save history during training
with open('../models/history1.pkl', 'wb') as f:
    pickle.dump(history1.history, f)

model1.save('../models/model_v1.keras')

In [ ]:
## plot results

plt.figure(figsize=(12, 5))

# accuracy
plt.subplot(1, 2, 1)
plt.plot(history1.history['accuracy'], label='train')
plt.plot(history1.history['val_accuracy'], label='val')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# loss
plt.subplot(1, 2, 2)
plt.plot(history1.history['loss'], label='train')
plt.plot(history1.history['val_loss'], label='val')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()

# test results
test_accuracy1, test_loss1 = model1.evaluate(X_test_normalized, y_test_cat, verbose=0)
print(f'Test Accuracy: {test_accuracy1:.4f} | Test Loss: {test_loss1:.4f}')

emotion_labels = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

# confusion matrix
y_pred = np.argmax(model1.predict(X_test_normalized), axis=1)
y_true = np.argmax(y_test_cat, axis=1)
cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=emotion_labels).plot(cmap='Blues', xticks_rotation='vertical')
plt.title('Confusion Matrix')
plt.show()

# classification report
print(classification_report(y_true, y_pred, target_names=emotion_labels))

Model 1 is overfitting, it does well on training data but poorly on validation data. Model is too simple/shallow, only 1 convolutional layer and 1 dense layer is not enough to learn and capture complexity of facial features.

The happy emotion was the easiest class to detect whereas disgust was completely missed. Since certain emotions have more images than others, we can possibly introduce class weights or perform data augmentation to address the class imbalance.

Overall accuracy = 41%, F1 score = 0.40

### Model 2 (Data Augmentation)

See how applying data augmentation affects the accuracy of the model.

In [ ]:
model2 = tf.keras.Sequential([
    layers.Input(shape=(48, 48, 1)),

    data_augmentation,

    # normalize pixels after augmentation
    layers.Rescaling(1./255),

    layers.Conv2D(32, (3, 3), activation='relu'),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    
    layers.Dense(7, activation='softmax')
])

model2.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history2 = model2.fit(
    X_train, y_train_cat,
    epochs=30,
    batch_size=64,
    validation_data=(X_val, y_val_cat),
    callbacks=[early_stop]
)

with open('../models/history2.pkl', 'wb') as f:
    pickle.dump(history2.history, f)

model2.save('./models/model_v2.keras')

In [ ]:
## plot results

plt.figure(figsize=(12, 5))

# accuracy
plt.subplot(1, 2, 1)
plt.plot(history2.history['accuracy'], label='train_accuracy')
plt.plot(history2.history['val_accuracy'], label='val_accuracy')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# loss
plt.subplot(1, 2, 2)
plt.plot(history2.history['loss'], label='train_loss')
plt.plot(history2.history['val_loss'], label='val_loss')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()

# test results
test_accuracy2, test_loss2 = model.evaluate(X_test, y_test_cat, verbose=0)
print(f'Test Accuracy: {test_accuracy2:.4f} | Test Loss: {test_loss2:.4f}')

emotion_labels = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

# confusion matrix
y_pred = np.argmax(model.predict(X_test), axis=1)
y_true = np.argmax(y_test_cat, axis=1)
cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=emotion_labels).plot(cmap='Blues', xticks_rotation='vertical')
plt.title('Confusion Matrix')
plt.show()

# classification report
print(classification_report(y_true, y_pred, target_names=emotion_labels))

Model 2 training and validation accuracies are close so only slightly overfitting. Adding data augmentation helped improve on the model but not by much as the digust class was still completely missed.

Overall accuracy = 43%, F1 score = 0.42